In [3]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q transformers
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q protobuf

print("=" * 60)
print("Libraries Installed Successfully")
print("=" * 60)

Libraries Installed Successfully


In [4]:
# ============================================================
# Import Libraries
# ============================================================

import torch

from transformers import (

    AutoTokenizer,

    AutoModelForCausalLM

)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

C:\Users\shyam\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries Imported Successfully


In [5]:
# ============================================================
# Check GPU
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("=" * 60)

print("Device :", device)

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

Device : cuda
GPU : NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# ============================================================
# Teacher Model
# ============================================================

TEACHER_MODEL = "Qwen/Qwen2.5-7B-Instruct"

teacher_tokenizer = AutoTokenizer.from_pretrained(

    TEACHER_MODEL

)

teacher_model = AutoModelForCausalLM.from_pretrained(

    TEACHER_MODEL,

    torch_dtype=torch.float16,

    device_map="auto"

)

print("=" * 60)
print("Teacher Model Loaded Successfully")
print("=" * 60)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TEACHER_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
print("Tokenizer loaded.")

print("Loading model...")
start = time.time()

teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Teacher model loaded!")
print(f"Time: {time.time() - start:.2f} seconds")

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    filename="model-00001-of-00002.safetensors",
)

print(path)

In [ ]:
# ============================================================
# Freeze Teacher
# ============================================================

for param in teacher_model.parameters():

    param.requires_grad = False

teacher_model.eval()

print("=" * 60)
print("Teacher Model Frozen")
print("=" * 60)

In [ ]:
# ============================================================
# Verify Teacher
# ============================================================

print("=" * 60)

print("Teacher Model :", TEACHER_MODEL)

print("Device :", next(teacher_model.parameters()).device)

print("Training Mode :", teacher_model.training)

trainable = sum(p.requires_grad for p in teacher_model.parameters())

print("Trainable Parameters :", trainable)

print("=" * 60)

In [ ]:
# ============================================================
# Test Teacher Inference
# ============================================================

prompt = "Explain pneumonia in one sentence."

inputs = teacher_tokenizer(

    prompt,

    return_tensors="pt"

).to(device)

with torch.no_grad():

    outputs = teacher_model.generate(

        **inputs,

        max_new_tokens=50,

        do_sample=False

    )

response = teacher_tokenizer.decode(

    outputs[0],

    skip_special_tokens=True

)

print("=" * 60)
print(response)
print("=" * 60)

In [3]:
from safetensors.torch import load_file

path = r"C:\Users\shyam\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct\snapshots\aa8e72537993ba99e69dfaafa59ed015b17504d1\model-00001-of-00002.safetensors"

print("Loading...")
weights = load_file(path)
print("Loaded", len(weights), "tensors")

Loading...
Loaded 264 tensors
